In [44]:
!pip install pandas torch numpy sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [61]:
import pandas as pd

df = pd.read_csv("heart_attack_prediction_dataset.csv")
print(df.head())
print(df.describe())

  Patient ID  Age     Sex  Cholesterol Blood Pressure  Heart Rate  Diabetes  \
0    BMW7812   67    Male          208         158/88          72         0   
1    CZE1114   21    Male          389         165/93          98         1   
2    BNI9906   21  Female          324         174/99          72         1   
3    JLN3497   84    Male          383        163/100          73         1   
4    GFO8847   66    Male          318          91/88          93         1   

   Family History  Smoking  Obesity  ...  Sedentary Hours Per Day  Income  \
0               0        1        0  ...                 6.615001  261404   
1               1        1        1  ...                 4.963459  285768   
2               0        0        0  ...                 9.463426  235282   
3               1        1        0  ...                 7.648981  125640   
4               1        1        1  ...                 1.514821  160555   

         BMI  Triglycerides  Physical Activity Days Per Week  

In [62]:
def retrieve_diastolic_blood_pressure(value):
    return value[value.find('/')+1:]

def retrieve_systolic_blood_pressure(value):
    return value[:value.find('/')]

df['Diastolic Blood Pressure'] = df['Blood Pressure'].map(retrieve_diastolic_blood_pressure)
df['Systolic Blood Pressure'] = df['Blood Pressure'].map(retrieve_systolic_blood_pressure)
print(df['Diastolic Blood Pressure'])
print(df['Systolic Blood Pressure'])

df = df.drop(columns=['Blood Pressure', 'Patient ID'])

0        88
1        93
2        99
3       100
4        88
       ... 
8758     76
8759    102
8760     75
8761     67
8762     67
Name: Diastolic Blood Pressure, Length: 8763, dtype: object
0       158
1       165
2       174
3       163
4        91
       ... 
8758     94
8759    157
8760    161
8761    119
8762    138
Name: Systolic Blood Pressure, Length: 8763, dtype: object


In [64]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

def create_preprocessor(df: pd.DataFrame, target_col: str):
    categorical = df.select_dtypes(include=["object"]).columns.tolist()

    if target_col in categorical:
        categorical.remove(target_col)

    for g in ["passed"]:
        if g in categorical:
            categorical.remove(g)

    numeric = df.select_dtypes(exclude=["object"]).columns.tolist()


    if target_col in numeric:
        numeric.remove(target_col)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
            ("num", StandardScaler(), numeric),
        ]
    )
    return preprocessor

RANDOM_STATE = 42
TEST_SIZE = int(df.__len__() * 0.2)
TARGET = 'Heart Attack Risk'

X = df.drop(columns=[TARGET])
y = df[TARGET]

train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
# print(train_x.info())
# print(train_y.info())

# print(test_x.info())
# print(test_y.info())


preprocessor = create_preprocessor(df, TARGET)

train_x_t = preprocessor.fit_transform(train_x)
test_x_t = preprocessor.transform(test_x)

le = LabelEncoder()
train_y_t = le.fit_transform(train_y)
test_y_t = le.transform(test_y)



In [ ]:
import torch
import torch.nn as nn

class DeepNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4):
        super(DeepNet, self).__init__()
        self.ff1 = nn.Linear(26, hidden_layer)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3),
        self.ff2 = nn.Linear(hidden_layer,1)
        

    def forward(self, x):
        x = self.ff1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.ff2(x)
        return x
    
x_train_t = torch.tensor(train_x_t.toarray() if hasattr(train_x_t, "toarray") else train_x_t, dtype=torch.float32)
x_test_t = torch.tensor(test_x_t.toarray() if hasattr(test_x_t, "toarray") else test_x_t, dtype=torch.float32)
y_train_t = torch.tensor(train_y_t, dtype=torch.long)
y_test_t = torch.tensor(test_y_t, dtype=torch.long)

model = DeepNet(x_train_t.shape[1], hidden_layer=x_train_t.shape[1] * 4)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

model.train()
with torch.no_grad():
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model(train_x_t)

        loss = criterion(outputs, train_y_t)
        loss.backward()
        optimizer.step()

model.eval()

    

TypeError: linear(): argument 'input' (position 1) must be Tensor, not csr_matrix